<a id="ridge"></a>

$$
\text{岭回归 (Ridge Regression, L2 正则化)} \\ \quad \\
\text{问题: 特征很多或者特征之间高度相关 (多重共线性) 时, } X^T X \text{ 接近奇异, 最小二乘解方差很大, 过拟合 (overfitting)} \\
\text{办法: 在损失里加一项惩罚, 不让 w 太大} \\
J(w) = \| y - X w \|^2 + \lambda \| w \|^2, \quad \lambda > 0 \\
\text{(截距不惩罚: 先把 X 的每一列和 y 都减去均值, 截距就单独等于 } \bar{y} - \bar{x}^T w \text{)} \\ \quad \\
\text{推导解析解: } \| w \|^2 = w^T w = w^T I w \text{, 由矩阵求导} \\
\nabla J = -2 X^T y + 2 X^T X w + 2 \lambda w = 0 \implies (X^T X + \lambda I) w = X^T y \\
w_{ridge} = (X^T X + \lambda I)^{-1} X^T y \\ \quad \\
\text{为什么一定可逆: } X^T X \text{ 半正定, 特征值 } \mu_i \geq 0 \text{, 加上 } \lambda I \text{ 后特征值变成 } \mu_i + \lambda > 0 \text{ (同一组特征向量)} \\
\text{所以 } X^T X + \lambda I \text{ 正定, 即使 } X^T X \text{ 奇异 (特征数比样本多, 或者完全共线) 也有唯一解}
$$

数学基础: [矩阵求导](../math_basics/linear_algebra.ipynb#matrix-calculus) · [正定矩阵](../math_basics/linear_algebra.ipynb#positive-definite) · [正规方程](linear_regression.ipynb#normal-equation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.linear_model import Ridge, Lasso, LinearRegression

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

rng = np.random.default_rng(0)
n, d = 40, 6
x_base = rng.normal(size=(n, 1))
X_raw = np.hstack([x_base + 0.01 * rng.normal(size=(n, 1)),     # 前两列几乎完全相同 (共线)
                   x_base + 0.01 * rng.normal(size=(n, 1)),
                   rng.normal(size=(n, d - 2))])
w_true = np.array([1.0, 1.0, 0.0, 2.0, 0.0, -1.5])
y = X_raw @ w_true + 3.0 + rng.normal(0, 0.5, n)

X_mean, y_mean = X_raw.mean(axis=0), y.mean()
Xc, yc = X_raw - X_mean, y - y_mean

def ridge_manual(Xc, yc, lam):
    return np.linalg.solve(Xc.T @ Xc + lam * np.eye(Xc.shape[1]), Xc.T @ yc)

lam = 5.0
w_r = ridge_manual(Xc, yc, lam)
b_r = y_mean - X_mean @ w_r
sk_r = Ridge(alpha=lam).fit(X_raw, y)
print(f'Ridge 手算: w = {w_r}, b = {b_r:.6f}')
print(f'sklearn   : w = {sk_r.coef_}, b = {sk_r.intercept_:.6f}, 一致? {np.allclose(w_r, sk_r.coef_) and np.isclose(b_r, sk_r.intercept_)}')

ols = LinearRegression().fit(X_raw, y)
eig = np.linalg.eigvalsh(Xc.T @ Xc)
print(f'X^T X 特征值 = {np.round(eig, 4)}, 条件数 = {eig.max() / eig.min():.0f}')
print(f'加 lambda 后条件数 = {(eig.max() + lam) / (eig.min() + lam):.2f}')
# 共线时 OLS 的系数方差很大: 固定 X, 重新生成 200 次噪声, 看前两个系数的波动
rng_rep = np.random.default_rng(1)
coef_ols, coef_ridge = [], []
for _ in range(200):
    y_rep = X_raw @ w_true + rng_rep.normal(0, 0.5, n)
    yc_rep = y_rep - y_rep.mean()
    coef_ols.append(np.linalg.solve(Xc.T @ Xc, Xc.T @ yc_rep)[:2])
    coef_ridge.append(ridge_manual(Xc, yc_rep, lam)[:2])
print(f'前两个系数的标准差 (200 次重复): OLS = {np.std(coef_ols, axis=0)}, Ridge = {np.std(coef_ridge, axis=0)}')
print(f'OLS 某一次的前两个系数 = {coef_ols[0]} (两个几乎相同的特征, 系数可以一个很大一个很小), Ridge = {coef_ridge[0]} (平分)')

<a id="ridge-svd"></a>

$$
\text{用 SVD 看 Ridge 在做什么: } X = U \Sigma V^T \ (\text{奇异值 } s_1 \geq s_2 \geq \cdots) \\
X^T X + \lambda I = V \Sigma^2 V^T + \lambda V V^T = V (\Sigma^2 + \lambda I) V^T, \quad X^T y = V \Sigma U^T y \\
w_{ridge} = V (\Sigma^2 + \lambda I)^{-1} \Sigma U^T y = \sum_i \frac{s_i}{s_i^2 + \lambda} v_i (u_i^T y) \\
\text{对比最小二乘 } (\lambda = 0) \text{: } w_{OLS} = \sum_i \frac{1}{s_i} v_i (u_i^T y) \quad (s_i \text{ 很小时 } \frac{1}{s_i} \text{ 爆炸}) \\ \quad \\
\text{预测值: } \hat{y} = X w_{ridge} = \sum_i u_i \underbrace{\frac{s_i^2}{s_i^2 + \lambda}}_{\text{收缩因子 } \in (0, 1)} (u_i^T y) \\
\text{奇异值大的方向 (信息多) 几乎不收缩, 奇异值小的方向 (噪声主导) 被压到接近 0} \\
\text{有效自由度 (Effective Degrees of Freedom): } df(\lambda) = \sum_i \frac{s_i^2}{s_i^2 + \lambda} \text{, } \lambda = 0 \text{ 时等于特征数 d, } \lambda \to \infty \text{ 时趋于 0}
$$

数学基础: [奇异值分解与条件数](../math_basics/linear_algebra.ipynb#svd) · [正交矩阵](../math_basics/linear_algebra.ipynb#orthogonal)

In [ ]:
U, s, Vt = np.linalg.svd(Xc, full_matrices=False)
w_svd = Vt.T @ (s / (s**2 + lam) * (U.T @ yc))
print(f'SVD 公式 == 正规方程形式? {np.allclose(w_svd, w_r)}')
print(f'奇异值 = {np.round(s, 4)}')
print(f'收缩因子 s^2 / (s^2 + lambda) = {np.round(s**2 / (s**2 + lam), 4)}')

lams = np.logspace(-3, 4, 60)
paths = np.array([ridge_manual(Xc, yc, l) for l in lams])
dfs = [np.sum(s**2 / (s**2 + l)) for l in lams]

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for j in range(d):
    axs[0].semilogx(lams, paths[:, j], label=f'w{j + 1}')
axs[0].set_xlabel('lambda')
axs[0].set_title('ridge coefficient path')
axs[1].semilogx(lams, dfs, label='effective df')
axs[1].set_xlabel('lambda')
plot_nicely()
plt.show()

<a id="map"></a>

$$
\text{贝叶斯角度: 正则化 = 最大后验估计 (Maximum A Posteriori, MAP)} \\ \quad \\
\text{贝叶斯公式: } p(w \mid y) = \frac{p(y \mid w) p(w)}{p(y)} \propto p(y \mid w) \, p(w) \quad (\text{后验} \propto \text{似然} \times \text{先验}) \\
\text{MLE 只最大化似然; MAP 最大化后验, 多了先验 } p(w) \text{ 这一项} \\ \quad \\
\text{似然: } y \mid w \sim N(X w, \sigma^2 I) \implies -\ln p(y \mid w) = \frac{1}{2\sigma^2} \| y - X w \|^2 + \text{const} \\ \quad \\
\text{1. 高斯先验 } w_j \sim N(0, \tau^2) \text{ (独立): } -\ln p(w) = \frac{1}{2\tau^2} \| w \|^2 + \text{const} \\
-\ln p(w \mid y) = \frac{1}{2\sigma^2} \| y - X w \|^2 + \frac{1}{2\tau^2} \| w \|^2 + \text{const} \\
\text{乘以 } 2\sigma^2 \text{ (不改变最小值点): } \| y - X w \|^2 + \frac{\sigma^2}{\tau^2} \| w \|^2 \implies \text{Ridge, } \lambda = \frac{\sigma^2}{\tau^2} \\
\text{先验越窄 (} \tau \text{ 越小, 越相信 w 接近 0), } \lambda \text{ 越大} \\ \quad \\
\text{2. 拉普拉斯先验 (Laplace) } p(w_j) = \frac{1}{2b} e^{-|w_j| / b} \text{: } -\ln p(w) = \frac{1}{b} \sum_j |w_j| + \text{const} \\
\| y - X w \|^2 + \frac{2\sigma^2}{b} \| w \|_1 \implies \text{Lasso (L1 正则化)} \\
\text{拉普拉斯分布在 0 处是尖的, 把更多的概率放在 0 附近, 所以 Lasso 的解会有很多正好等于 0 的系数}
$$

数学基础: [贝叶斯公式](../math_basics/statistics.ipynb#bayes) · [最大似然估计](../math_basics/statistics.ipynb#mle) · [正态分布](../math_basics/statistics.ipynb#normal-distribution)

In [ ]:
sigma_noise, tau_prior = 0.5, 0.25
lam_map = sigma_noise**2 / tau_prior**2

def neg_log_posterior(w):
    return np.sum((yc - Xc @ w)**2) / (2 * sigma_noise**2) + np.sum(w**2) / (2 * tau_prior**2)

w_map = minimize(neg_log_posterior, np.zeros(d), method='BFGS', options={'gtol': 1e-10}).x
print(f'lambda = sigma^2 / tau^2 = {lam_map}')
print(f'MAP (数值最小化负对数后验) = {w_map}')
print(f'Ridge 解析解               = {ridge_manual(Xc, yc, lam_map)}, 一致? {np.allclose(w_map, ridge_manual(Xc, yc, lam_map), atol=1e-5)}')

wg = np.linspace(-4, 4, 400)
plt.figure(figsize=(4, 3))
plt.plot(wg, np.exp(-wg**2 / 2) / np.sqrt(2 * np.pi), label='Gaussian prior -> L2')
plt.plot(wg, np.exp(-np.abs(wg)) / 2, label='Laplace prior -> L1')
plt.xlabel('w')
plot_nicely()
plt.show()

<a id="lasso"></a>

$$
\text{Lasso (L1 正则化) 与坐标下降 (Coordinate Descent)} \\
J(w) = \frac{1}{2n} \| y - X w \|^2 + \alpha \| w \|_1 \quad (\text{sklearn 的写法, 和上面只差一个常数倍}) \\
|w_j| \text{ 在 0 处不可导, 没有解析解. 坐标下降: 每次固定其他坐标, 只优化一个 } w_j \\ \quad \\
\text{记去掉第 j 个特征后的残差 } r_j = y - \sum_{k \neq j} x_k w_k \text{, 只看 } w_j \text{ 的部分:} \\
J(w_j) = \frac{1}{2n} \| r_j - x_j w_j \|^2 + \alpha |w_j| + \text{const} \\
\text{记 } \rho_j = \frac{1}{n} x_j^T r_j, \ z_j = \frac{1}{n} x_j^T x_j \text{, 展开: } J(w_j) = \frac{z_j}{2} w_j^2 - \rho_j w_j + \alpha |w_j| + \text{const} \\ \quad \\
\text{分情况求导 (每段都是二次函数):} \\
w_j > 0 \text{: } z_j w_j - \rho_j + \alpha = 0 \implies w_j = \frac{\rho_j - \alpha}{z_j} \text{, 要求 } w_j > 0 \iff \rho_j > \alpha \\
w_j < 0 \text{: } z_j w_j - \rho_j - \alpha = 0 \implies w_j = \frac{\rho_j + \alpha}{z_j} \text{, 要求 } \rho_j < -\alpha \\
|\rho_j| \leq \alpha \text{: 上面两种都不成立, 最小值只能在不可导点 } w_j = 0 \\
\text{(在 0 处左导数 } -\rho_j - \alpha \leq 0 \text{, 右导数 } -\rho_j + \alpha \geq 0 \text{, 确实是最小值)} \\ \quad \\
w_j = \frac{S(\rho_j, \alpha)}{z_j}, \quad S(\rho, \alpha) = \text{sign}(\rho) \max(|\rho| - \alpha, 0) \quad (\text{软阈值 Soft-Thresholding}) \\
\text{只要 } |\rho_j| \leq \alpha \text{ (这个特征和残差的相关性不够强), 系数就被精确地设为 0 → 自动做特征选择}
$$

In [ ]:
def soft_threshold(rho, alpha):
    return np.sign(rho) * np.maximum(np.abs(rho) - alpha, 0.0)

def lasso_cd(Xc, yc, alpha, n_iter=5000, tol=1e-12):
    n_s, d_s = Xc.shape
    w = np.zeros(d_s)
    z = (Xc**2).sum(axis=0) / n_s
    for _ in range(n_iter):
        w_old = w.copy()
        for j in range(d_s):
            r_j = yc - Xc @ w + Xc[:, j] * w[j]           # 去掉第 j 个特征的贡献
            rho = Xc[:, j] @ r_j / n_s
            w[j] = soft_threshold(rho, alpha) / z[j]
        if np.max(np.abs(w - w_old)) < tol:
            break
    return w

alpha = 0.1
w_l = lasso_cd(Xc, yc, alpha)
sk_l = Lasso(alpha=alpha, tol=1e-12, max_iter=100000).fit(X_raw, y)
print(f'Lasso 坐标下降: w = {w_l + 0.0}')
print(f'sklearn Lasso : w = {sk_l.coef_ + 0.0}, 一致? {np.allclose(w_l, sk_l.coef_, atol=1e-6)}')
print(f'等于 0 的系数个数: Lasso = {np.sum(w_l == 0)}, Ridge = {np.sum(w_r == 0)}')

# 坐标下降每一步都确实是一维最小值: 和暴力搜索对比
j = 3
r_j = yc - Xc @ w_l + Xc[:, j] * w_l[j]
grid = np.linspace(-4, 4, 80001)
obj = ((r_j[:, None] - Xc[:, [j]] * grid)**2).sum(axis=0) / (2 * n) + alpha * np.abs(grid)
print(f'w_{j + 1}: 软阈值公式 = {w_l[j]:.5f}, 暴力搜索 = {grid[np.argmin(obj)]:.5f}')

alphas = np.logspace(-3, 0.5, 50)
lasso_paths = np.array([lasso_cd(Xc, yc, a, n_iter=2000, tol=1e-8) for a in alphas])
plt.figure(figsize=(4, 3))
for j in range(d):
    plt.semilogx(alphas, lasso_paths[:, j], label=f'w{j + 1}')
plt.xlabel('alpha')
plt.title('lasso path: coefficients hit exactly 0')
plot_nicely()
plt.show()

<a id="bias-variance"></a>

$$
\text{偏差-方差分解 (Bias-Variance Decomposition): 为什么正则化能降低测试误差} \\ \quad \\
\text{真实关系 } y = f(x) + \epsilon, \ E[\epsilon] = 0, \ Var(\epsilon) = \sigma^2 \text{, } \hat{f} \text{ 是在随机训练集上训练得到的模型 (所以 } \hat{f}(x) \text{ 是随机变量)} \\
\text{在新的点 x 上的期望平方误差, 记 } \bar{f} = E[\hat{f}(x)] \text{:} \\
y - \hat{f} = \underbrace{(f - \bar{f})}_{\text{常数}} + \underbrace{(\bar{f} - \hat{f})}_{\text{均值为 0}} + \underbrace{\epsilon}_{\text{均值为 0, 和训练集独立}} \\
E[(y - \hat{f})^2] = (f - \bar{f})^2 + E[(\bar{f} - \hat{f})^2] + E[\epsilon^2] + \underbrace{2(f - \bar{f}) E[\bar{f} - \hat{f}] + 2(f - \bar{f}) E[\epsilon] + 2 E[(\bar{f} - \hat{f})\epsilon]}_{\text{交叉项全部为 0}} \\
E[(y - \hat{f}(x))^2] = \underbrace{(f(x) - E[\hat{f}(x)])^2}_{\text{偏差}^2 \text{ Bias}^2} + \underbrace{Var(\hat{f}(x))}_{\text{方差 Variance}} + \underbrace{\sigma^2}_{\text{不可约误差}} \\ \quad \\
\lambda \text{ 小: 模型灵活, 偏差小但对训练集很敏感, 方差大 (过拟合)} \\
\lambda \text{ 大: 系数被压向 0, 方差小但偏差大 (欠拟合)} \\
\text{测试误差关于 } \lambda \text{ 是 U 形, 中间某个 } \lambda \text{ 最好}
$$

数学基础: [期望和方差的性质](../math_basics/statistics.ipynb#expectation-variance-properties) · [方差](../math_basics/statistics.ipynb#variance)

In [ ]:
rng_bv = np.random.default_rng(42)
f_true = lambda x: np.sin(2 * np.pi * x)
sigma_bv, n_train, degree, n_sets = 0.3, 20, 9, 500
x_test = np.linspace(0.05, 0.95, 50)

def poly_features(x, mean=None, std=None):
    F = np.column_stack([x**k for k in range(1, degree + 1)])
    if mean is None:
        mean, std = F.mean(axis=0), F.std(axis=0)
    return (F - mean) / std, mean, std

lams_bv = np.logspace(-8, 1, 25)
bias2, variance, test_err = [], [], []
x_sets = rng_bv.uniform(0, 1, size=(n_sets, n_train))
noise_sets = rng_bv.normal(0, sigma_bv, size=(n_sets, n_train))
for l in lams_bv:
    preds = np.empty((n_sets, len(x_test)))
    for s_i in range(n_sets):
        x_tr = x_sets[s_i]
        y_tr = f_true(x_tr) + noise_sets[s_i]
        F_tr, m, sd = poly_features(x_tr)
        w_bv = ridge_manual(F_tr, y_tr - y_tr.mean(), l)
        F_te, _, _ = poly_features(x_test, m, sd)
        preds[s_i] = F_te @ w_bv + y_tr.mean()
    b2 = np.mean((f_true(x_test) - preds.mean(axis=0))**2)
    var = np.mean(preds.var(axis=0))
    # 直接模拟新的测试标签, 验证分解: E[(y - f_hat)^2] = bias^2 + var + sigma^2
    y_new = f_true(x_test) + rng_bv.normal(0, sigma_bv, size=preds.shape)
    bias2.append(b2); variance.append(var); test_err.append(np.mean((y_new - preds)**2))

bias2, variance, test_err = map(np.array, (bias2, variance, test_err))
k_mid = 12
print(f'lambda = {lams_bv[k_mid]:.2e}: bias^2 + var + sigma^2 = {bias2[k_mid] + variance[k_mid] + sigma_bv**2:.4f}, 模拟的测试误差 = {test_err[k_mid]:.4f}')
best = np.argmin(test_err)
print(f'测试误差最小的 lambda = {lams_bv[best]:.2e}, 此时 bias^2 = {bias2[best]:.4f}, var = {variance[best]:.4f}')

plt.figure(figsize=(4, 3))
plt.loglog(lams_bv, bias2, label='bias^2')
plt.loglog(lams_bv, variance, label='variance')
plt.loglog(lams_bv, bias2 + variance + sigma_bv**2, 'k', label='bias^2 + var + sigma^2')
plt.loglog(lams_bv, test_err, 'r--', label='simulated test error')
plt.xlabel('lambda')
plot_nicely()
plt.show()